# 🧭 OptiCompromise: Herramienta de Programación de Compromiso Multiobjetivo Continuo
### Asignatura: Introducción al Análisis de Decisiones (IAD) | Facultad de Minas – Universidad Nacional de Colombia
**Docente:** Patricia Jaramillo Álvarez, PhD | **Apoyo Académico:** Jaime Andrés Díaz Guerra, MSc
**Autores:** Fabián Andrés Hurtado Arango (1036450768) & Oscar David Arango Garcia (1011591946)

---
## Presentación y Objetivo
Esta herramienta implementa la metodología de **Programación de Compromiso (Compromise Programming)** para problemas continuos con múltiples objetivos en conflicto. Permite calcular la **Matriz de Pagos**, el **Punto Ideal**, el **Punto Anti-Ideal (Nadir aproximado)**, y resolver de manera interactiva las soluciones compromiso bajo las métricas:
- **Métrica $ (Manhattan):** Programación Lineal con efecto compensatorio.
- **Métrica $ (Euclidiana):** Programación Cuadrática Convexa con compromiso suave.
- **Métrica \infty$ (Tchebysheff / Minimax):** Programación Lineal minimax que garantiza equilibrio y equidad estricta entre objetivos.

Adicionalmente, aproxima y visualiza la **Frontera Eficiente de Pareto** y permite variar los pesos interactivamente con deslizadores.


In [ ]:
# 1. Instalación e importación de bibliotecas científicas necesarias
import sys
import numpy as np
import pandas as pd
from scipy.optimize import linprog, minimize
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

print("✓ Bibliotecas importadas exitosamente. SciPy version:", sys.modules["scipy"].__version__)


In [ ]:
# 3. Modelos de Ejemplo Precargados del Curso
MODELOS_EJEMPLO = {
    "1. Adquisición Flota Eléctrica (6 Autos, 4 Objetivos - Taller IAD)": {
        "name": "Adquisición de Flota de Vehículos Eléctricos (Caso Taller IAD)",
        "variables": [
            {"name": "x1 (BYD Seal)", "lower_bound": 0.0, "upper_bound": 50.0},
            {"name": "x2 (Tesla Model 3)", "lower_bound": 0.0, "upper_bound": 50.0},
            {"name": "x3 (Hyundai Ioniq 5)", "lower_bound": 0.0, "upper_bound": 50.0},
            {"name": "x4 (Kia EV6)", "lower_bound": 0.0, "upper_bound": 50.0},
            {"name": "x5 (Volvo EX30)", "lower_bound": 0.0, "upper_bound": 50.0},
            {"name": "x6 (Chevrolet Equinox EV)", "lower_bound": 0.0, "upper_bound": 50.0}
        ],
        "objectives": [
            {"name": "Costo Flota (USD)", "sense": "minimize", "coefficients": [44000.0, 46500.0, 49000.0, 48000.0, 42000.0, 43500.0]},
            {"name": "Autonomía Flota (km)", "sense": "maximize", "coefficients": [570.0, 630.0, 520.0, 540.0, 480.0, 510.0]},
            {"name": "Tiempo Carga (min)", "sense": "minimize", "coefficients": [32.0, 27.0, 18.0, 18.0, 26.0, 30.0]},
            {"name": "Seguridad Flota (pts)", "sense": "maximize", "coefficients": [9.0, 9.5, 9.6, 9.7, 10.0, 9.3]}
        ],
        "constraints": [
            {"name": "Tamaño Flota", "coefficients": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0], "sense": "==", "rhs": 50.0},
            {"name": "Maletero Min (L)", "coefficients": [400.0, 594.0, 527.0, 520.0, 318.0, 748.0], "sense": ">=", "rhs": 24000.0},
            {"name": "Presupuesto Max (USD)", "coefficients": [44000.0, 46500.0, 49000.0, 48000.0, 42000.0, 43500.0], "sense": "<=", "rhs": 2350000.0}
        ]
    },
    "2. Elección Biobjetivo: BYD vs Tesla (2 Autos, 2 Objetivos)": {
        "name": "Elección Biobjetivo: BYD Seal vs Tesla Model 3",
        "variables": [
            {"name": "x1 (BYD Seal)", "lower_bound": 0.0, "upper_bound": 50.0},
            {"name": "x2 (Tesla Model 3)", "lower_bound": 0.0, "upper_bound": 50.0}
        ],
        "objectives": [
            {"name": "Costo Flota (USD)", "sense": "minimize", "coefficients": [44000.0, 46500.0]},
            {"name": "Autonomía Flota (km)", "sense": "maximize", "coefficients": [570.0, 630.0]}
        ],
        "constraints": [
            {"name": "Tamaño Flota", "coefficients": [1.0, 1.0], "sense": "==", "rhs": 50.0},
            {"name": "Presupuesto Max (USD)", "coefficients": [44000.0, 46500.0], "sense": "<=", "rhs": 2300000.0}
        ]
    }
}
print("✓ Modelos del caso de Carros Eléctricos (Taller IAD) cargados correctamente.")


In [ ]:
# 4. Interfaz Interactiva de Usuario (Widgets Reactivos en Colab)
dropdown_modelo = widgets.Dropdown(
    options=list(MODELOS_EJEMPLO.keys()),
    description="Modelo:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="550px")
)

slider_w1 = widgets.FloatSlider(
    value=0.30, min=0.0, max=1.0, step=0.05,
    description="Ponderación w1:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px")
)

output_dashboard = widgets.Output()

def actualizar_dashboard(*args):
    with output_dashboard:
        clear_output(wait=True)
        m_data = MODELOS_EJEMPLO[dropdown_modelo.value]
        n_objs = len(m_data["objectives"])
        
        if n_objs == 2:
            w1 = slider_w1.value
            w_list = [w1, 1.0 - w1]
        else:
            w_list = [0.30, 0.25, 0.25, 0.20]
        
        res = resolver_opticompromise(m_data, weights=w_list)
        
        # Mostrar Matriz de Pagos e Ideal
        obj_names = [obj['name'] for obj in m_data['objectives']]
        row_labels = ['Optimo ' + name for name in obj_names]
        df_payoff = pd.DataFrame(res['payoff'], index=row_labels, columns=obj_names)
        df_payoff.loc['Punto Ideal (Z*)'] = res['ideal']
        df_payoff.loc['Punto Anti-Ideal / Nadir'] = res['anti_ideal']
        
        display(HTML('<h3>📊 1. Matriz de Pagos y Puntos de Referencia</h3>'))
        display(df_payoff.round(2))
        
        # Calcular resultados de las 3 métricas
        def evaluar(x):
            z_vals = [float(np.dot(obj['coefficients'], x)) for obj in m_data['objectives']]
            d_vals = res['alpha'] @ x + res['beta']
            ach_pct = 100.0 * (1.0 - d_vals)
            return z_vals, ach_pct
            
        z_l1, ach_l1 = evaluar(res['L1_x'])
        z_l2, ach_l2 = evaluar(res['L2_x'])
        z_linf, ach_linf = evaluar(res['Linf_x'])
        
        var_names = [v['name'] for v in m_data['variables']]
        
        def format_sol_vars(x_sol):
            active = [var_names[i] + ': ' + f'{x_sol[i]:.1f}' for i in range(len(x_sol)) if abs(x_sol[i]) > 0.05]
            return ', '.join(active) if active else 'Todas en 0'
            
        row_data = []
        metrics = [('L1 (Manhattan - Lineal)', z_l1, ach_l1, res['L1_x'], 'Efecto compensatorio (vértice extremo)'),
                   ('L2 (Euclidiana - Cuadrática)', z_l2, ach_l2, res['L2_x'], 'Compromiso suave (distancia euclídea mínima)'),
                   ('L_inf (Tchebysheff - Minimax)', z_linf, ach_linf, res['Linf_x'], 'Equilibrio y equidad estricta entre criterios')]
        
        for name, z_v, ach, x_v, prop in metrics:
            item = {'Métrica': name, 'Asignación Flota (x*)': format_sol_vars(x_v)}
            for idx, oname in enumerate(obj_names):
                item[oname + ' (Real)'] = round(z_v[idx], 1)
                item[oname + ' (% Logro)'] = f'{ach[idx]:.2f}%'
            item['Propiedad Clave'] = prop
            row_data.append(item)
            
        df_results = pd.DataFrame(row_data)
        display(HTML('<h3>🎯 2. Comparativa de Soluciones Compromiso</h3>'))
        display(df_results)
        
        # Gráfica según número de objetivos
        if n_objs == 2:
            plt.figure(figsize=(9, 5))
            plt.scatter(res['ideal'][0], res['ideal'][1], color='gold', s=200, marker='*', label='Punto Ideal', zorder=5)
            plt.scatter(res['anti_ideal'][0], res['anti_ideal'][1], color='crimson', s=120, marker='X', label='Anti-Ideal (Nadir)', zorder=5)
            plt.scatter(z_l1[0], z_l1[1], color='royalblue', s=140, marker='s', label=f'L1 (w1={w_list[0]:.2f}, w2={w_list[1]:.2f})')
            plt.scatter(z_l2[0], z_l2[1], color='forestgreen', s=140, marker='o', label=f'L2 (w1={w_list[0]:.2f}, w2={w_list[1]:.2f})')
            plt.scatter(z_linf[0], z_linf[1], color='purple', s=140, marker='D', label=f'L_inf (w1={w_list[0]:.2f}, w2={w_list[1]:.2f})')
            plt.title('Espacio de Objetivos: ' + m_data['name'], fontsize=13, fontweight='bold')
            plt.xlabel(obj_names[0], fontsize=11)
            plt.ylabel(obj_names[1], fontsize=11)
            plt.legend(loc='best', frameon=True)
            plt.grid(True, linestyle='--', alpha=0.6)
            plt.tight_layout()
            plt.show()
        else:
            # Gráfico Radar de % Logro para múltiples criterios
            categories = obj_names
            N = len(categories)
            angles = [n / float(N) * 2 * np.pi for n in range(N)]
            angles += angles[:1]
            
            fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
            plt.xticks(angles[:-1], categories, size=10, fontweight='bold')
            ax.set_rlabel_position(0)
            plt.yticks([25, 50, 75, 100], ['25%', '50%', '75%', '100%'], color='grey', size=8)
            plt.ylim(0, 110)
            
            # Ideal (100%)
            val_ideal = [100] * (N + 1)
            ax.plot(angles, val_ideal, linewidth=2, linestyle='--', color='gold', label='Ideal (100%)')
            
            # L1
            val_l1 = list(ach_l1) + [ach_l1[0]]
            ax.plot(angles, val_l1, linewidth=2, color='royalblue', label='L1 (Manhattan)')
            ax.fill(angles, val_l1, 'royalblue', alpha=0.1)
            
            # L2
            val_l2 = list(ach_l2) + [ach_l2[0]]
            ax.plot(angles, val_l2, linewidth=2, color='forestgreen', label='L2 (Euclidiana)')
            ax.fill(angles, val_l2, 'forestgreen', alpha=0.1)
            
            # Linf
            val_linf = list(ach_linf) + [ach_linf[0]]
            ax.plot(angles, val_linf, linewidth=2.5, color='purple', label='L_inf (Minimax)')
            ax.fill(angles, val_linf, 'purple', alpha=0.15)
            
            plt.title('Perfil Multicriterio (% Logro): ' + m_data['name'], size=13, fontweight='bold', y=1.08)
            plt.legend(loc='upper right', bbox_to_anchor=(1.25, 1.1))
            plt.tight_layout()
            plt.show()

dropdown_modelo.observe(actualizar_dashboard, names='value')
slider_w1.observe(actualizar_dashboard, names='value')

display(HTML('<h2>🚀 Panel de Control Interactivo OptiCompromise</h2>'))
display(dropdown_modelo)
display(slider_w1)
display(output_dashboard)

# Ejecución inicial
actualizar_dashboard()


In [ ]:
# 4. Interfaz Interactiva de Usuario (Widgets Reactivos en Colab)
dropdown_modelo = widgets.Dropdown(
    options=list(MODELOS_EJEMPLO.keys()),
    description="Modelo:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px")
)

slider_w1 = widgets.FloatSlider(
    value=0.5, min=0.0, max=1.0, step=0.05,
    description="Peso Objetivo 1 (w1):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px")
)

output_dashboard = widgets.Output()

def actualizar_dashboard(*args):
    with output_dashboard:
        clear_output(wait=True)
        m_data = MODELOS_EJEMPLO[dropdown_modelo.value]
        w1 = slider_w1.value
        w2 = 1.0 - w1
        
        res = resolver_opticompromise(m_data, weights=[w1, w2])
        
        # Mostrar Matriz de Pagos e Ideal
        df_payoff = pd.DataFrame(res["payoff"], 
                                 index=[f"Optimo {obj['name']}" for obj in m_data["objectives"]],
                                 columns=[obj["name"] for obj in m_data["objectives"]])
        df_payoff.loc["Punto Ideal (Z*)"] = res["ideal"]
        df_payoff.loc["Punto Anti-Ideal / Nadir"] = res["anti_ideal"]
        
        display(HTML("<h3>📊 1. Matriz de Pagos y Puntos de Referencia</h3>"))
        display(df_payoff.round(2))
        
        # Calcular resultados de las 3 métricas
        def evaluar(x):
            z_vals = [float(np.dot(obj["coefficients"], x)) for obj in m_data["objectives"]]
            d_vals = res["alpha"] @ x + res["beta"]
            ach_pct = 100.0 * (1.0 - d_vals)
            return z_vals, ach_pct
            
        z_l1, ach_l1 = evaluar(res["L1_x"])
        z_l2, ach_l2 = evaluar(res["L2_x"])
        z_linf, ach_linf = evaluar(res["Linf_x"])
        
        obj_names = [obj["name"] for obj in m_data["objectives"]]
        df_results = pd.DataFrame({
            "Métrica": ["L1 (Manhattan - Lineal)", "L2 (Euclidiana - Cuadrática)", "L_inf (Tchebysheff - Minimax)"],
            f"{obj_names[0]}": [z_l1[0], z_l2[0], z_linf[0]],
            f"{obj_names[1]}": [z_l1[1], z_l2[1], z_linf[1]],
            "Grado Logro Obj 1 (%)": [f"{ach_l1[0]:.2f}%", f"{ach_l2[0]:.2f}%", f"{ach_linf[0]:.2f}%"],
            "Grado Logro Obj 2 (%)": [f"{ach_l1[1]:.2f}%", f"{ach_l2[1]:.2f}%", f"{ach_linf[1]:.2f}%"],
            "Propiedad Clave": [
                "Efecto compensatorio (vértice extremo)",
                "Compromiso suave (curva elíptica)",
                "Equilibrio estricto (Z1' = Z2' si pesos iguales)"
            ]
        })
        
        display(HTML("<h3>🎯 2. Comparativa de Soluciones Compromiso</h3>"))
        display(df_results)
        
        # Gráfica del Espacio de Objetivos
        plt.figure(figsize=(9, 5))
        # Punto ideal y anti-ideal
        plt.scatter(res["ideal"][0], res["ideal"][1], color="gold", s=200, marker="*", label="Punto Ideal", zorder=5)
        plt.scatter(res["anti_ideal"][0], res["anti_ideal"][1], color="crimson", s=120, marker="X", label="Anti-Ideal (Nadir)", zorder=5)
        
        # Puntos compromiso
        plt.scatter(z_l1[0], z_l1[1], color="royalblue", s=140, marker="s", label=f"L1 (w1={w1:.2f}, w2={w2:.2f})")
        plt.scatter(z_l2[0], z_l2[1], color="forestgreen", s=140, marker="o", label=f"L2 (w1={w1:.2f}, w2={w2:.2f})")
        plt.scatter(z_linf[0], z_linf[1], color="purple", s=140, marker="D", label=f"L_inf (w1={w1:.2f}, w2={w2:.2f})")
        
        plt.title(f"Espacio de Objetivos: {m_data['name']}", fontsize=13, fontweight="bold")
        plt.xlabel(obj_names[0], fontsize=11)
        plt.ylabel(obj_names[1], fontsize=11)
        plt.legend(loc="best", frameon=True)
        plt.grid(True, linestyle="--", alpha=0.6)
        plt.tight_layout()
        plt.show()

dropdown_modelo.observe(actualizar_dashboard, names="value")
slider_w1.observe(actualizar_dashboard, names="value")

display(HTML("<h2>🚀 Panel de Control Interactivo OptiCompromise</h2>"))
display(dropdown_modelo)
display(slider_w1)
display(output_dashboard)

# Ejecución inicial
actualizar_dashboard()
